In [3]:
from google.colab import files

uploaded = files.upload()

Saving code1_plan_results.json to code1_plan_results.json


In [5]:
# ════════════════════════════════════════════════════════════
#  CODE 2: Plan → SQL  |  Evaluate SQL Token F1
#  Upload: lora_adapter_plan2sql.zip, code1_plan_results.json
#  (run Code 1 first to get code1_plan_results.json)
# ════════════════════════════════════════════════════════════

!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

import json, zipfile, os, re, gc, time
from collections import Counter
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── Config ────────────────────────────────────────────────────
ADAPTER_ZIP  = "./lora_adapter_plan2sql (1).zip"           # plan2sql weights
ADAPTER_DIR  = "./lora_adapter_plan2sql"
INPUT_PATH   = "./code1_plan_results.json"             # output from Code 1
OUTPUT_PATH  = "./code2_sql_results.json"
MAX_INPUT    = 384
MAX_TARGET   = 384

# ── Unzip ─────────────────────────────────────────────────────
if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── Load model ────────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-xl"
dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto",
)
print("Loading plan2sql LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Model ready!")

# ── Load data ─────────────────────────────────────────────────
with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples")

# ── Generate SQL ──────────────────────────────────────────────
def generate_sql(plan_text):
    inp = tokenizer(plan_text, return_tensors="pt",
                    max_length=MAX_INPUT, truncation=True)
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=MAX_TARGET,
                             num_beams=4, early_stopping=True, length_penalty=1.0)
    return tokenizer.decode(out[0], skip_special_tokens=True)

start = time.time()
for i, r in enumerate(data):
    r["pred_sql"] = generate_sql(r["pred_plan"])
    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (len(data) - i - 1)
        print(f"  [{i+1}/{len(data)}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── Token F1 ──────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pt = tokenize(pred); gt = tokenize(gold)
    if len(pt) == 0 and len(gt) == 0: return 1.0
    if len(pt) == 0 or  len(gt) == 0: return 0.0
    overlap   = sum((Counter(pt) & Counter(gt)).values())
    precision = overlap / len(pt)
    recall    = overlap / len(gt)
    if precision + recall == 0: return 0.0
    return 2 * precision * recall / (precision + recall)

f1_scores = []
for r in data:
    f1 = token_f1(r["pred_sql"], r["gold_sql"])
    f1_scores.append(f1)
    r["sql_f1"] = round(f1, 4)

avg_f1 = sum(f1_scores) / len(f1_scores)
print("\n══════════════════════════════════════")
print(f"   CODE 2: SQL EVAL (n={len(data)})")
print("══════════════════════════════════════")
print(f"  SQL Token F1 : {avg_f1*100:.2f}%")
print("══════════════════════════════════════")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"Saved → {OUTPUT_PATH}")

print("\n── Samples ──")
for i in [0, 10, 50]:
    r = data[i]
    print(f"\n[{i}] SQL F1={r['sql_f1']:.3f}")
    print(f"  PRED PLAN : {r['pred_plan']}")
    print(f"  GOLD SQL  : {r['gold_sql']}")
    print(f"  PRED SQL  : {r['pred_sql']}")

Extracted to: ./lora_adapter_plan2sql
Loading base model (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading plan2sql LoRA...
Model ready!
Loaded 101 samples
  [25/101]  elapsed: 0.9 min  ETA: 2.7 min
  [50/101]  elapsed: 2.7 min  ETA: 2.7 min
  [75/101]  elapsed: 3.7 min  ETA: 1.3 min
  [100/101]  elapsed: 5.5 min  ETA: 0.1 min

══════════════════════════════════════
   CODE 2: SQL EVAL (n=101)
══════════════════════════════════════
  SQL Token F1 : 92.32%
══════════════════════════════════════
Saved → ./code2_sql_results.json

── Samples ──

[0] SQL F1=1.000
  PRED PLAN : step1: SCAN | table: product || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  GOLD SQL  : SELECT count(*) FROM product
  PRED SQL  : SELECT count(*) FROM product

[10] SQL F1=1.000
  PRED PLAN : step1: SCAN | table: orders || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute SUM(Total_Amount) || step3: PROJECT | columns: SUM(Total_Amount)
  GOLD SQL  : SELECT sum(Total_Amount) FROM orders
  PRED SQL  : SELECT sum(Total_Amount) FROM orders

[